In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path

REVISION = 'a10cc1512eabd3dde888204e902eca88bddb4951'
FINAL_HASH = '0f9c156c5fd85773bef61488d0978c731e3c7d44ed6697b4fb226f36ddb8ffb3'
BANK_HASH = 'd9867a2498753ec62df21e24987de41ae1a4addbc83006eed051ce1d50626bed'
GENERATION_HASH = '44911f2968297a54d41087abdaebf4bf3cbff94221497a299281e6202afccfd2'
BASE_HASH = 'fb03af823a1184ff1078c28b0c3d99f71f4e707c101fc53c05516101910c2e91'
BEST_HASH = 'd9d55724c72543f33d92c5c36e29377463da6ee65218ba77eeba178844e3c215'
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()

runs = []
for path in Path('/kaggle/input').rglob('run.json'):
    if path.parent.name != 'persona_lora_run':
        continue
    run = json.loads(path.read_text())
    if run.get('checkpoints', {}).get('final', {}).get('adapter_model.safetensors', {}).get('sha256') == FINAL_HASH:
        runs.append(path)
assert len(runs) == 1, 'Attach the completed persona run with the exact final adapter'
TRAIN = runs[0].parent
INPUT = TRAIN.parent
run = json.loads(runs[0].read_text())
assert run['status'] == 'completed' and run['step'] == 13 and run['revision'] == REVISION
assert run['frozen_base_sha256_before'] == run['frozen_base_sha256_after']
for kind in ['best', 'final']:
    for name, meta in run['checkpoints'][kind].items():
        path = TRAIN/(kind+'_adapter')/name
        assert path.stat().st_size == meta['bytes'] and sha(path) == meta['sha256'], str(path)
SOURCE = INPUT/'persona_source'
assert sha(SOURCE/'generate_persona.py') == GENERATION_HASH
assert sha(SOURCE/'persona_eval_40.json') == BANK_HASH
OLD = INPUT/'persona_evaluation'
assert sha(OLD/'baseline/outputs.json') == BASE_HASH
assert sha(OLD/'best_adapter/outputs.json') == BEST_HASH
for kind in ['baseline', 'best_adapter']:
    meta = json.loads((OLD/kind/'run.json').read_text())
    assert meta['status'] == 'completed' and meta['completed'] == 40 and meta['revision'] == REVISION
    assert meta['bank_sha256'] == BANK_HASH and meta['precision'] == 'float32' and meta['seed'] == 42
    assert meta['decoding'] == {'do_sample':False,'max_new_tokens':256,'repetition_penalty':1.1,'no_repeat_ngram_size':0}
assert json.loads((OLD/'best_adapter/run.json').read_text())['adapter_sha256'] == run['checkpoints']['best']['adapter_model.safetensors']['sha256']
ROOT = Path('/kaggle/working/persona_final_review')
assert not ROOT.exists(), 'Never overwrite a previous evaluation'
ROOT.mkdir()
shutil.copyfile(SOURCE/'generate_persona.py', ROOT/'generate_persona.py')
shutil.copyfile(SOURCE/'persona_eval_40.json', ROOT/'persona_eval_40.json')
for kind in ['baseline', 'best_adapter']:
    shutil.copytree(OLD/kind, ROOT/kind)
(ROOT/'source.json').write_text(json.dumps({'source_training_version':357420502,'final_update':13,'final_adapter_sha256':FINAL_HASH,'bank_sha256':BANK_HASH,'generation_sha256':GENERATION_HASH,'input_path':str(INPUT)},indent=2)+'\n')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
ENV = Path('/kaggle/working/persona_eval_env')
subprocess.run([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(ENV)],check=True)
PYTHON = str(ENV/'bin/python')
subprocess.run([sys.executable,'-m','pip','--python',PYTHON,'install','--quiet','transformers==4.51.3','peft==0.15.2','accelerate==1.7.0','datasets==3.6.0','trl==0.19.1'],check=True)
print('FINAL PERSONA SOURCE AND ALL CHECKPOINT FILES VERIFIED',flush=True)


In [ ]:
subprocess.run([PYTHON,str(ROOT/'generate_persona.py'),'--bank',str(ROOT/'persona_eval_40.json'),'--output',str(ROOT/'final_adapter'),'--revision',REVISION,'--adapter',str(TRAIN/'final_adapter')],check=True)
final_meta = json.loads((ROOT/'final_adapter/run.json').read_text())
assert final_meta['status']=='completed' and final_meta['completed']==40 and final_meta['adapter_sha256']==FINAL_HASH
assert final_meta['bank_sha256']==BANK_HASH and final_meta['decoding']==json.loads((ROOT/'baseline/run.json').read_text())['decoding']
answers = {name:json.loads((ROOT/name/'outputs.json').read_text()) for name in ['baseline','best_adapter','final_adapter']}
bank = json.loads((ROOT/'persona_eval_40.json').read_text())
for name, rows in answers.items():
    assert len(rows)==40 and len({r['id'] for r in rows})==40
    for row, prompt in zip(rows,bank):
        assert all(row[k]==prompt[k] for k in ['id','instruction','category'])
    assert sha(ROOT/name/'outputs.json')==json.loads((ROOT/name/'run.json').read_text())['outputs_sha256']

import random
summary = {'status':'completed','training_version':357420502,'final_update':13,'final_adapter_sha256':FINAL_HASH,'count':40,'final_blanks':final_meta['blanks'],'final_length_limits':sum(r['finish_reason']=='length' for r in answers['final_adapter']),'comparisons':{}}
for other in ['baseline','best_adapter']:
    changed = [a['id'] for a,b in zip(answers['final_adapter'],answers[other]) if a['output']!=b['output']]
    summary['comparisons'][other] = {'identical':40-len(changed),'changed_ids':changed}
    rng = random.Random(42)
    pairs = []
    mapping = []
    for a,b in zip(answers[other],answers['final_adapter']):
        swap = rng.choice([False,True])
        pairs.append({'id':a['id'],'instruction':a['instruction'],'response_a':b['output'] if swap else a['output'],'response_b':a['output'] if swap else b['output']})
        mapping.append({'id':a['id'],'category':a['category'],'a':'final_adapter' if swap else other,'b':other if swap else 'final_adapter'})
    folder = ROOT/('pairs-final-vs-'+other)
    folder.mkdir()
    (folder/'pairs.jsonl').write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in pairs))
    (folder/'hidden_mapping.json').write_text(json.dumps(mapping,indent=2)+'\n')
(ROOT/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print('FINAL PERSONA REVIEW GENERATION COMPLETED '+json.dumps(summary),flush=True)
